<a href="https://colab.research.google.com/github/RossIsland/MINLP-Surrogate-Modelling/blob/main/Gas_Compressor_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CELL 1: CORE DEPENDENCY PROVISIONING
# ==============================================================================
!pip install -q torch torchvision pyomo scikit-learn pandas numpy
!pip install -q amplpy ampltools
!python -m amplpy.modules install cbc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.1 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip install -i https://pypi.ampl.com ampl_module_base ampl_module_cbc
Looking in indexes: https://pypi.ampl.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 46.0 MB/s eta 0:00:00
Imported ampl_module_base.
Imported ampl_module_base.
Imported ampl_module_cbc.


In [ ]:
# ==============================================================================
# CELL 2: GAS TURBINE COMPRESSOR THERMODYNAMIC SIMULATION & CORRELATIONS
# ==============================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

def simulate_gas_turbine_compressor(P_in, T_in, Q_in, s, n):
    """
    Rigorously models a Gas Turbine Driven Centrifugal Compressor Station.
    Inputs:
      P_in  : Inlet Pressure (MPa)
      T_in  : Inlet Temperature (K)
      Q_in  : Volumetric Flow Rate (10^4 m3/d)
      s     : Turbine Rotational Speed (r/min)
      n     : Active Parallel Operating Units
    """
    # Constant Natural Gas Physical Properties
    k = 1.31               # Isentropic exponent (heat capacity ratio for CH4)
    R = 518.3              # Specific gas constant (J/kg·K)
    Z = 0.92               # Average compressibility factor
    LHV = 3.6e7            # Lower Heating Value of fuel gas (J/m3)
    rho_std = 0.717        # Standard gas density (kg/m3)

    # 1. Distribute flow evenly through the active parallel compressor units
    n_active = max(float(n), 1.0)
    q_per_unit = Q_in / n_active
    speed_ratio = s / 5000.0  # Normalized against nominal speed design point

    # 2. Centrifugal Compressor Performance Characteristic Head Model (Affinity Laws)
    # Head rises quadratically with speed, drops quadratically with increased throughput
    head_max = 65000.0 * (speed_ratio**2)
    head_loss = 0.015 * (q_per_unit**2)
    H_isen = max(head_max - head_loss, 15000.0)

    # 3. Invert Isentropic Equation to find true Outlet Pressure (MPa)
    pressure_ratio = (1.0 + (H_isen * (k - 1)) / (k * Z * R * T_in))**(k / (k - 1))
    P_out = P_in * pressure_ratio

    # 4. Compute Discharge Temperature using polytropic/isentropic relationship
    # Assumes a 78% polytropic compressor efficiency component
    eta_c = 0.78
    T_out = T_in * (1.0 + (1.0 / eta_c) * ((P_out / P_in)**((k - 1) / k) - 1.0))

    # 5. Mass Flow Calculation (Convert 10^4 m3/day to real kg/s)
    mass_flow = (Q_in * 10000.0 * rho_std) / 86400.0

    # 6. Shaft Power Requirement (kW)
    W_shaft = (mass_flow * H_isen) / (eta_c * 1000.0)

    # 7. Gas Turbine Fuel Gas Consumption Flow Rate Profile (10^4 m3/d)
    # Turbine thermal efficiency drops off design point if engine speed varies from nominal
    eta_t = 0.34 * (1.0 - 0.05 * (speed_ratio - 1.0)**2)
    fuel_energy_flow = (W_shaft * 1000.0) / eta_t        # Fuel energy flow input in Watts (J/s)
    fg_m3_per_s = fuel_energy_flow / LHV                 # Fuel volume flow rate in m3/s
    fg_comp6 = (fg_m3_per_s * 86400.0) / 10000.0         # Scale parameter directly to 10^4 m3/d

    # 8. Boundary Envelopes (Surge and Choke Constraints)
    Q_min = 120.0 * speed_ratio * n_active
    Q_max = 1800.0 * speed_ratio * n_active

    # Standard security clipping limits
    P_out = np.clip(P_out, P_in + 0.2, 12.0)
    T_out = np.clip(T_out, 273.15, 380.0)

    return P_out, T_out, Q_in, fg_comp6, Q_min, Q_max

# Generate Dataset
np.random.seed(42)
num_samples = 10000

P_in_s = np.random.uniform(3.0, 7.5, num_samples)
T_in_s = np.random.uniform(280.0, 315.0, num_samples)
Q_in_s = np.random.uniform(400.0, 2500.0, num_samples)
s_s    = np.random.uniform(3000.0, 6000.0, num_samples)
n_s    = np.random.choice([1, 2, 3, 4], size=num_samples)

po, to, qo, fg, qmin, qmax = [], [], [], [], [], []
for i in range(num_samples):
    p_o, t_o, q_o, f_g, q_i, q_a = simulate_gas_turbine_compressor(P_in_s[i], T_in_s[i], Q_in_s[i], s_s[i], n_s[i])
    po.append(p_o)
    to.append(t_o)
    qo.append(q_o)
    fg.append(f_g)
    qmin.append(q_i)
    qmax.append(q_a)

df = pd.DataFrame({
    'P_in': P_in_s, 'T_in': T_in_s, 'Q_in': Q_in_s, 's': s_s, 'n': n_s,
    'P_out': po, 'T_out': to, 'Q_out': qo, 'fg_comp6': fg, 'Q_min': qmin, 'Q_max': qmax
})

X = df[['P_in', 'T_in', 'Q_in', 's', 'n']].values
Y = df[['P_out', 'T_out', 'Q_out', 'fg_comp6', 'Q_min', 'Q_max']].values

X_offset, X_factor = X.mean(axis=0), X.std(axis=0)
Y_offset, Y_factor = Y.mean(axis=0), Y.std(axis=0)

X_scaled = (X - X_offset) / X_factor
Y_scaled = (Y - Y_offset) / Y_factor

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y_scaled, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
Y_test_t  = torch.tensor(Y_test, dtype=torch.float32)

print("Gas Turbine Compressor Dataset Built! Shape:", df.shape)

Gas Turbine Compressor Dataset Built! Shape: (10000, 11)


In [ ]:
# ==============================================================================
# CELL 3: TRAINING THE GAS TURBINE DEEP REACTIVATION LAYER NETWORK
# ==============================================================================
class GasCompressorANN(nn.Module):
    def __init__(self):
        super(GasCompressorANN, self).__init__()
        self.fc1 = nn.Linear(5, 20)   # 5 physical inputs, 20 hidden nodes
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(20, 6)   # 6 targets: P_out, T_out, Q_out, fg_comp6, Q_min, Q_max

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = GasCompressorANN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

print("Optimizing Weight Tensors against Gas Compressor Thermodynamics...")
for epoch in range(120):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train_t)
    loss = criterion(predictions, Y_train_t)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    test_loss = criterion(model(X_test_t), Y_test_t)
    print(f"Validated Network Physics Congruence - Test MSE Loss: {test_loss.item():.6f}")

W1 = model.fc1.weight.detach().numpy()
b1 = model.fc1.bias.detach().numpy()
W2 = model.fc2.weight.detach().numpy()
b2 = model.fc2.bias.detach().numpy()

Optimizing Weight Tensors against Gas Compressor Thermodynamics...
Validated Network Physics Congruence - Test MSE Loss: 0.034900


In [ ]:
# ==============================================================================
# CELL 4: LOCAL PIECEWISE LAYER FEASIBILITY RESOLUTION VIA PYOMO
# ==============================================================================
from pyomo.environ import *
from amplpy import modules

opt_model = ConcreteModel()

# Continuous optimization tracking nodes for Gas Turbine Workspace
opt_model.P_in  = Var(bounds=(3.0, 7.5), initialize=5.5)
opt_model.T_in  = Var(bounds=(280.0, 315.0), initialize=298.15)
opt_model.Q_in  = Var(bounds=(400.0, 2500.0), initialize=1200.0)
opt_model.s     = Var(bounds=(3000.0, 6000.0), initialize=4500.0)
opt_model.n     = Var(bounds=(1.0, 4.0), initialize=2.0)

opt_model.P_out = Var(bounds=(3.0, 12.0))
opt_model.T_out = Var(bounds=(273.15, 380.0))
opt_model.Q_out = Var(bounds=(0.0, 3000.0))
opt_model.fg_comp6 = Var(bounds=(0.0, 50.0)) # Fuel Gas constraint allocation tracking variable
opt_model.Q_min = Var(bounds=(0.0, 1000.0))
opt_model.Q_max = Var(bounds=(0.0, 4000.0))

opt_model.neurons = RangeSet(0, 19)
opt_model.u       = Var(opt_model.neurons, bounds=(0.0, 100.0))
opt_model.delta   = Var(opt_model.neurons, within=Binary)

M_ub, M_lb = 250.0, -250.0

def get_scaled_input(m):
    return [
        (m.P_in - X_offset[0]) / X_factor[0],
        (m.T_in - X_offset[1]) / X_factor[1],
        (m.Q_in - X_offset[2]) / X_factor[2],
        (m.s    - X_offset[3]) / X_factor[3],
        (m.n    - X_offset[4]) / X_factor[4]
    ]

def relu_bigm_rules(m, k):
    inputs = get_scaled_input(m)
    u_hat = sum(W1[k, i] * inputs[i] for i in range(5)) + b1[k]
    yield m.u[k] >= u_hat
    yield m.u[k] <= u_hat - M_lb * (1 - m.delta[k])
    yield m.u[k] <= M_ub * m.delta[k]
    yield u_hat  >= M_lb * (1 - m.delta[k])

opt_model.relu_constraints = ConstraintList()
for k in opt_model.neurons:
    for c in relu_bigm_rules(opt_model, k):
        opt_model.relu_constraints.add(c)

def output_rule(m, out_idx, var_target):
    y_scaled = sum(W2[out_idx, k] * m.u[k] for k in m.neurons) + b2[out_idx]
    return var_target == (y_scaled * Y_factor[out_idx]) + Y_offset[out_idx]

opt_model.out_p_con    = Constraint(rule=lambda m: output_rule(m, 0, m.P_out))
opt_model.out_t_con    = Constraint(rule=lambda m: output_rule(m, 1, m.T_out))
opt_model.out_q_con    = Constraint(rule=lambda m: output_rule(m, 2, m.Q_out))
opt_model.out_fg_con   = Constraint(rule=lambda m: output_rule(m, 3, m.fg_comp6))
opt_model.out_qmin_con = Constraint(rule=lambda m: output_rule(m, 4, m.Q_min))
opt_model.out_qmax_con = Constraint(rule=lambda m: output_rule(m, 5, m.Q_max))

# Fix current operating scenario conditions to resolve activation configuration
opt_model.P_in.fix(5.2)
opt_model.T_in.fix(293.15)
opt_model.Q_in.fix(1450.0)
opt_model.s.fix(4800.0)
opt_model.n.fix(2.0)

opt_model.obj = Objective(expr=opt_model.fg_comp6, sense=minimize)
solver = SolverFactory("cbcnl", executable=modules.find("cbc"), solve_io="nl")
results = solver.solve(opt_model)

print("\n--- Feasibility Pass Successful ---")
print(f"Target Fuel Flow Output (fg_comp6): {value(opt_model.fg_comp6):.4f} 10^4 m3/d")
print(f"Discharge Pressure Profile        : {value(opt_model.P_out):.4f} MPa")


--- Feasibility Pass Successful ---
Target Fuel Flow Output (fg_comp6): 5.1155 10^4 m3/d
Discharge Pressure Profile        : 7.3385 MPa


In [ ]:
# ==============================================================================
# CELL 5: THERMODYNAMIC UNIT-CORRECTED EXTRACTION LOOP FOR AMPL
# ==============================================================================
active_neurons = [k for k in opt_model.neurons if value(opt_model.delta[k]) > 0.5]
output_names = ["pressure_ratio", "temperature_ratio", "volumetric_outflow", "fuel_consumption", "surge_boundary", "choke_boundary"]
target_vars  = ["P_node[16]", "T_node[16]", "Q_pipe[10]", "fg_comp6", "Q_min[6]", "Q_max[6]"]

print("="*80)
print("UNIT-CORRECTED THERMODYNAMIC EQUATIONS FOR GAS COMPRESSOR STATION 6")
print("="*80)
print("Variables mapping context:")
print("  x[0]=P_node[15], x[1]=T_node[15], x[2]=Q_comp[6], x[3]=s_comp[6], x[4]=n_comp[6]\n")

# Keep at 1.0 because the network is trained directly on daily scale metrics (10^4 m3/d)
flow_conversion = 1.0

for idx, name in enumerate(output_names):
    m_scaled = np.zeros(5)
    c_scaled = b2[idx]

    for k in active_neurons:
        m_scaled += W2[idx, k] * W1[k, :]
        c_scaled += W2[idx, k] * b1[k]

    # Reconvert from standard normal neural space back to physical coordinates
    m_phys = (m_scaled / X_factor) * Y_factor[idx]
    c_phys = Y_offset[idx] + Y_factor[idx] * (c_scaled - sum((m_scaled * X_offset) / X_factor))

    # Unit correction alignment loops
    m_phys[2] = m_phys[2] * flow_conversion
    if name in ["volumetric_outflow", "fuel_consumption", "surge_boundary", "choke_boundary"]:
        m_phys = m_phys / flow_conversion
        c_phys = c_phys / flow_conversion

    print(f"subject to comp_6_{name}:")
    print(f"    {target_vars[idx]} = ({m_phys[0]:.6e} * P_node[15]) + ")
    print(f"                        ({m_phys[1]:.6e} * T_node[15]) + ")
    print(f"                        ({m_phys[2]:.6e} * Q_comp[6]) + ")
    print(f"                        ({m_phys[3]:.6e} * s_comp[6]) + ")
    print(f"                        ({m_phys[4]:.6e} * n_comp[6]) + ({c_phys:.6f});\n")
print("="*80)

UNIT-CORRECTED THERMODYNAMIC EQUATIONS FOR GAS COMPRESSOR STATION 6
Variables mapping context:
  x[0]=P_node[15], x[1]=T_node[15], x[2]=Q_comp[6], x[3]=s_comp[6], x[4]=n_comp[6]

subject to comp_6_pressure_ratio:
    P_node[16] = (1.702710e+00 * P_node[15]) + 
                        (-7.819046e-03 * T_node[15]) + 
                        (-8.932528e-04 * Q_comp[6]) + 
                        (1.216046e-03 * s_comp[6]) + 
                        (4.405679e-01 * n_comp[6]) + (-4.646351);

subject to comp_6_temperature_ratio:
    T_node[16] = (1.164226e+00 * P_node[15]) + 
                        (1.225983e+00 * T_node[15]) + 
                        (-1.409495e-02 * Q_comp[6]) + 
                        (1.752047e-02 * s_comp[6]) + 
                        (7.171396e+00 * n_comp[6]) + (-119.330046);

subject to comp_6_volumetric_outflow:
    Q_pipe[10] = (-2.305059e+01 * P_node[15]) + 
                        (-1.274703e+00 * T_node[15]) + 
                        (1.233040e+00 * Q_comp